# Книга Excel: `read_excel` на своём компьютере

Блокнот к уроку «Книга Excel: листы, шапка, итоги» главы 3. В браузере курса те же примеры работают прямо в уроке; здесь — те же шаги в своём Jupyter на книге `zerno_finance.xlsx`, плюс запись результата обратно в Excel.

**Что нужно.** Библиотека openpyxl: в папке проекта из урока про uv выполните в терминале `uv add openpyxl` и перезапустите ядро (Kernel → Restart Kernel). В распакованном курсе книга лежит в `../../data/`. Если скачали блокнот отдельно, положите книгу рядом с ним или в папку `data/`.

**Как работать.** Выполняйте ячейки сверху вниз. В конце — три упражнения: место для кода помечено ✎, под каждым — ячейка самопроверки.

In [1]:
from pathlib import Path
import pandas as pd

folders = (Path("."), Path("data"), Path("../../data"), Path("../data"))
BOOK = next((folder / "zerno_finance.xlsx" for folder in folders
             if (folder / "zerno_finance.xlsx").is_file()), None)
if BOOK is None:
    raise FileNotFoundError("Положите zerno_finance.xlsx рядом с блокнотом или в папку data/.")
MONTHS = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн", "Июл", "Авг", "Сен"]

try:
    import openpyxl
    print("pandas", pd.__version__, "| openpyxl", openpyxl.__version__, "| книга:", BOOK)
except ImportError:
    print("Нет openpyxl. Выполните в терминале: uv add openpyxl — и перезапустите ядро.")

pandas 3.0.6 | openpyxl 3.1.5 | книга: data/zerno_finance.xlsx


## 1. Какие листы в книге

`sheet_name=None` читает все листы сразу и возвращает словарь «имя листа → таблица». Удобно, чтобы осмотреться.

In [2]:
book = pd.read_excel(BOOK, sheet_name=None)
{name: df.shape for name, df in book.items()}

{'Описание': (2, 1), 'План': (11, 16), 'Факт': (12, 14)}

## 2. Лист «План»: пропустить шапку

Сначала — как есть. Заголовок листа стал названием первого столбца, остальные — `Unnamed`.

In [3]:
raw = pd.read_excel(BOOK, sheet_name="План")
raw.head(4)

,Сеть кофеен «Зерно»,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,"План выручки на 2026 год, тыс. ₽",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Код,Кофейня,Город,Янв,Фев,Мар,Апр,Май,Июн,Июл,Авг,Сен,Окт,Ноя,Дек,Год
3,PRM-1,Ленина,Пермь,1194.6,1144.4,1217.9,1138.8,1122.2,1079.8,1006.9,1051.6,1212.9,1249.7,1272.3,1396.1,14087.2


In [4]:
def header_row(sheet):
    first = pd.read_excel(BOOK, sheet_name=sheet, header=None, usecols=[0]).iloc[:, 0]
    positions = first.index[first.eq("Код")]
    if len(positions) != 1:
        raise ValueError(f"Лист {sheet}: ожидали одну строку с заголовком Код")
    return int(positions[0])

plan = pd.read_excel(BOOK, sheet_name="План", skiprows=header_row("План"))
print(plan.dtypes.value_counts())
plan

float64    13
str         3
Name: count, dtype: int64


,Код,Кофейня,Город,Янв,Фев,Мар,Апр,Май,Июн,Июл,Авг,Сен,Окт,Ноя,Дек,Год
0,PRM-1,Ленина,Пермь,1194.6,1144.4,1217.9,1138.8,1122.2,1079.8,1006.9,1051.6,1212.9,1249.7,1272.3,1396.1,14087.2
1,PRM-2,Компрос,Пермь,756.3,734.4,776.7,749.9,725.5,680.4,651.0,668.1,780.9,833.9,821.5,885.6,9064.2
2,KZN-1,Баумана,Казань,1363.7,1288.2,1385.6,1310.9,1257.7,1223.8,1164.6,1199.7,1413.8,1451.1,1473.5,1565.0,16097.6
3,KZN-2,Кремлёвская,Казань,815.0,802.9,829.7,819.5,789.5,726.0,708.7,740.9,849.2,875.9,910.5,971.5,9839.3
4,TMN-1,Республики,Тюмень,894.3,863.1,884.0,875.3,820.7,798.2,763.4,785.4,898.4,946.0,978.5,1051.5,10558.8
5,UFA-1,Ленина,Уфа,709.2,675.9,722.2,673.2,672.1,632.4,614.0,617.7,720.5,766.2,769.9,834.0,8407.3
6,EKB-1,Вайнера,Екатеринбург,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,513.0,1054.2,1139.4,2706.6
7,Итого,NaN,NaN,5733.1,5508.9,5816.1,5567.6,5387.7,5140.6,4908.6,5063.4,5875.7,6635.8,7280.4,7843.1,70761.0


Названия столбцов на месте, месяцы — числа. В отличие от CSV, `decimal=","` не нужен: Excel хранит числа как числа, запятая — только способ их показать.

## 3. Лист «Факт»: шапка выше, столбцы буквами

На исходном листе «Факт» над таблицей четыре строки. Найдём заголовок по слову «Код», чтобы служебная строка в новой выгрузке не сдвинула таблицу. Столбцы выбираем буквами, как при выделении в Excel: A — код, D:L — январь–сентябрь.

In [5]:
fact = pd.read_excel(BOOK, sheet_name="Факт", skiprows=header_row("Факт"), usecols="A,D:L")
fact

,Код,Янв,Фев,Мар,Апр,Май,Июн,Июл,Авг,Сен
0,PRM-1,1232.7,1162.8,1244.2,1144.7,1173.3,1103.6,1073.1,1058.2,1214.7
1,PRM-2,706.3,737.3,767.3,736.7,704.5,675.7,614.3,656.3,754.9
2,KZN-1,1387.6,1349.7,1422.2,1410.9,1346.8,1308.0,1205.3,1273.0,1463.3
3,KZN-2,734.4,718.3,454.6,742.7,736.4,629.8,660.0,650.5,736.0
4,TMN-1,873.2,833.8,900.9,880.7,782.3,817.7,748.3,786.3,896.1
5,UFA-1,686.3,638.1,696.8,626.8,625.1,607.5,555.4,585.2,692.3
6,EKB-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Итого,5620.5,5440.0,5486.0,5542.5,5368.4,5142.3,4856.4,5009.5,5757.3


## 4. Строка «Итого»: убрать и сверить

Сумма по кофейням должна совпасть с итоговой строкой. `assert` остановит блокнот, если это не так.

In [6]:
body = fact[fact["Код"] != "Итого"]
total = fact[fact["Код"] == "Итого"]
by_shops = round(body[MONTHS].sum().sum(), 1)
from_total = round(total[MONTHS].sum().sum(), 1)
assert abs(by_shops - from_total) < 0.5, "сумма по кофейням не сходится с «Итого»"
print("факт за 9 месяцев:", by_shops, "тыс. ₽ — сходится с «Итого»")

факт за 9 месяцев: 48222.9 тыс. ₽ — сходится с «Итого»


## 5. План против факта

In [7]:
plan_months = plan[plan["Код"] != "Итого"].set_index("Код")[MONTHS]
fact_months = body.set_index("Код")[MONTHS]
p = plan_months.sum(axis=1)
f = fact_months.sum(axis=1)
plan_count = plan_months.count(axis=1)
fact_count = fact_months.count(axis=1)
complete = (plan_count == len(MONTHS)) & (fact_count == len(MONTHS)) & (p > 0)
done = (100 * f[complete] / p[complete]).round(1).sort_values()
done

Код
KZN-2     85.6
UFA-1     94.6
PRM-2     97.4
TMN-1     99.2
PRM-1    102.3
KZN-1    104.8
dtype: float64

## 6. Сохранить отчёт обратно в Excel

В сводке сохраняем все кофейни. Рядом с суммой внесённых значений показываем, сколько месяцев заполнено. Для сравнения выполнения нужны полный период и положительный план. Проверьте строку EKB-1 в сохранённой книге: чем она отличается от строки с девятью явно внесёнными нулями?

`pd.ExcelWriter` пишет две таблицы на разные листы одной книги. Файл появится рядом с блокнотом.

In [8]:
report = pd.DataFrame({"сумма внесённого плана, тыс. ₽": p,
                       "сумма внесённого факта, тыс. ₽": f,
                       "месяцев плана": plan_count, "месяцев факта": fact_count,
                       "выполнение, %": done})
with pd.ExcelWriter("plan_fact_report.xlsx") as writer:
    report.to_excel(writer, sheet_name="Выполнение")
    body.to_excel(writer, sheet_name="Факт без итога", index=False)
list(pd.read_excel("plan_fact_report.xlsx", sheet_name=None))

['Выполнение', 'Факт без итога']

## Упражнения

**1.** Прочитайте лист «Факт» не через `skiprows`, а через `header=` — так, чтобы получилась та же таблица `fact` (те же столбцы A, D:L). Результат — в `fact_h`.

In [9]:
fact_h = pd.read_excel(BOOK,sheet_name="Факт",header=header_row("Факт"),usecols="A,D:L")
assert fact_h.equals(fact)
print("header и skiprows совпали:",fact_h.equals(fact))

header и skiprows совпали: True


In [10]:
# Самопроверка доступна после решения.


**2.** Прочитайте **все** листы книги одним вызовом и сохраните в `sheets` список их имён в том порядке, в каком они идут в книге.

In [11]:
sheets = list(pd.read_excel(BOOK,sheet_name=None))
print(sheets)

['Описание', 'План', 'Факт']


In [12]:
# Самопроверка доступна после решения.


**3.** В каких месяцах сумма внесённого факта по сети оказалась меньше 97 % суммы внесённого плана? Посчитайте отношение двух сумм по каждому месяцу и сохраните названия в `weak` («Янв», «Фев», …). Пустые ячейки не дают вклада в эти суммы. Чтобы сделать вывод о выполнении плана всей сетью, дополнительно сверили бы работающие в этом месяце точки с перечнем ожидаемых и проверили заполненность их данных.

In [13]:
monthly_done = 100*fact_months.sum()/plan_months.sum()
weak = monthly_done[monthly_done<97].index.tolist()
print("Меньше 97%:",weak)

Меньше 97%: ['Мар']


In [14]:
# Самопроверка доступна после решения.


## Моя проверка файла и альтернативного сценария
Пустые ячейки и нули имеют разный охват. Проверяю сохранённую книгу повторным чтением.

In [15]:
back = pd.read_excel('plan_fact_report.xlsx',sheet_name=None)
print('Две вкладки:',list(back))
pd.testing.assert_frame_equal(back['Выполнение'].set_index('Код'),report,check_dtype=False)
pd.testing.assert_frame_equal(back['Факт без итога'],body.reset_index(drop=True),check_dtype=False)
print('EKB-1 в сохранённом отчёте:')
print(back['Выполнение'].set_index('Код').loc[['EKB-1']])
# Контрфакт A: девять нулей при девяти положительных значениях плана.
p_alt = plan_months.copy()
f_alt = fact_months.copy()
p_alt.loc['EKB-1',MONTHS] = 100
f_alt.loc['EKB-1',MONTHS] = 0
valid_alt = p_alt.count(axis=1).eq(9) & f_alt.count(axis=1).eq(9) & p_alt.sum(axis=1).gt(0)
done_alt = (100*f_alt.sum(axis=1)[valid_alt]/p_alt.sum(axis=1)[valid_alt]).round(1)
assert done_alt.loc['EKB-1']==0
print('Альтернатива A: охват 9/9, план 900, факт 0, выполнение',done_alt.loc['EKB-1'])
# Контрфакт B: одна записанная нулевая сумма пропала, остаток sum() тот же.
f_alt.loc['EKB-1','Мар'] = float('nan')
valid_missing = p_alt.count(axis=1).eq(9) & f_alt.count(axis=1).eq(9) & p_alt.sum(axis=1).gt(0)
assert not valid_missing.loc['EKB-1']
print('Альтернатива B: внесённая сумма',f_alt.loc['EKB-1'].sum(),'; охват',f_alt.loc['EKB-1'].count(),'/9; рейтинг исключает')
import json
Path('excel-evidence.json').write_text(json.dumps({'sheets':sheets,'weak':weak,'report_sheets':list(back),'reread_equal':True,'ekb_original':{'plan_count':int(plan_count.loc['EKB-1']),'fact_count':int(fact_count.loc['EKB-1']),'plan_sum':float(p.loc['EKB-1']),'fact_sum':float(f.loc['EKB-1']),'in_rating':bool(complete.loc['EKB-1'])},'alternative_complete_pct':float(done_alt.loc['EKB-1']),'alternative_missing_count':int(f_alt.loc['EKB-1'].count())},ensure_ascii=False,indent=2))

Две вкладки: ['Выполнение', 'Факт без итога']
EKB-1 в сохранённом отчёте:
       сумма внесённого плана, тыс. ₽  сумма внесённого факта, тыс. ₽  \
Код                                                                     
EKB-1                             0.0                             0.0   

       месяцев плана  месяцев факта  выполнение, %  
Код                                                 
EKB-1              0              0            NaN  
Альтернатива A: охват 9/9, план 900, факт 0, выполнение 0.0
Альтернатива B: внесённая сумма 0.0 ; охват 8 /9; рейтинг исключает


377

**Вывод.** В исходной EKB-1 внесено 0 из 9 месяцев плана и факта; суммы внесённого нулевые, выручка периода неизвестна. При 9 нулях факта и положительном полном плане получилось бы 0% выполнения. Потеря одного нуля оставляет сумму 0, но охват 8/9 и исключает рейтинг. Только Мар ниже 97% по внесённым данным сети; это не доказательство полного охвата сети.